# 05 Model Comparison: IEEE-CIS Fraud Detection

Это `Jupyter notebook` для следующего шага после `03_baseline_model.ipynb` и `04_behavioral_features.ipynb`.

Цель:
- сравнить текущий честный baseline на `LogisticRegression` с еще одной простой моделью;
- проверить, есть ли в данных полезный нелинейный сигнал;
- не усложнять решение раньше времени и сохранить MVP-подход;
- интерпретировать результат не только по `roc_auc`, но и по anti-fraud метрикам.


## План работы

1. Загрузить `train_transaction` и `train_identity`.
2. Собрать тот же честный baseline feature set.
3. Разделить данные на `train / validation`.
4. Обучить `LogisticRegression` как reference model.
5. Обучить еще одну простую модель для сравнения.
6. Сравнить `precision`, `recall`, `f1`, `roc_auc` и `manual_review_rate`.
7. Коротко зафиксировать, какая модель полезнее для anti-fraud MVP.


## Что мы изучаем

На этом шаге мы изучаем не новые признаки, а разницу между простыми моделями.

Что важно понять:
- может ли нелинейная модель поймать сигнал, который не ловит `LogisticRegression`;
- улучшаются ли метрики без слишком сильного роста `manual_review_rate`;
- стоит ли двигаться дальше в сторону model comparison или сначала усиливать feature set.


In [6]:
from pathlib import Path
import warnings

from IPython.display import Markdown, display

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


In [7]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
TRANSACTION_PATH = DATA_DIR / 'train_transaction.csv'
IDENTITY_PATH = DATA_DIR / 'train_identity.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRANSACTION_PATH exists:', TRANSACTION_PATH.exists())
print('IDENTITY_PATH exists:', IDENTITY_PATH.exists())


PROJECT_ROOT: /Users/drhtka/Downloads/Projects/Llm_ml_RAG/anti_fraud_analytics_platform
TRANSACTION_PATH exists: True
IDENTITY_PATH exists: True


In [8]:
def load_csv_if_exists(path: Path):
    if path.exists():
        print(f'Loaded: {path.name}')
        return pd.read_csv(path)
    print(f'File not found: {path}')
    return None


transactions = load_csv_if_exists(TRANSACTION_PATH)
identity = load_csv_if_exists(IDENTITY_PATH)


Loaded: train_transaction.csv
Loaded: train_identity.csv


## Hypothesis For Fair Model Comparison

На этом шаге мы пока не обучаем модели.

Сначала собираем один и тот же честный baseline feature set, чтобы потом сравнение
`LogisticRegression` и следующей простой модели было корректным.

Что важно понять:

- обе модели должны обучаться на одном и том же наборе признаков;
- обе модели должны использовать один и тот же `train / validation split`;
- train-only статистики должны считаться только по train части.

Почему это важно:

- если модели сравниваются на разных данных или разных признаках,
  вывод будет нечестным;
- сначала нужна правильная основа, потом уже сравнение качества.

In [9]:
high_risk_p_domains = {'outlook.com'}
high_risk_r_domains = {'outlook.com', 'icloud.com', 'gmail.com'}

feature_df = transactions[
    [
        'TransactionID',
        'isFraud',
        'TransactionAmt',
        'ProductCD',
        'card1',
        'card4',
        'card6',
        'P_emaildomain',
        'R_emaildomain',
        'addr1',
        'TransactionDT',
    ]
].copy()

feature_df['feat_productcd_c_flag'] = (feature_df['ProductCD'] == 'C').astype(int)
feature_df['feat_card4_discover_flag'] = (feature_df['card4'] == 'discover').astype(int)
feature_df['feat_card6_credit_flag'] = (feature_df['card6'] == 'credit').astype(int)
feature_df['feat_high_risk_p_email_flag'] = feature_df['P_emaildomain'].isin(high_risk_p_domains).astype(int)
feature_df['feat_high_risk_r_email_flag'] = feature_df['R_emaildomain'].isin(high_risk_r_domains).astype(int)
feature_df['feat_missing_r_email_flag'] = feature_df['R_emaildomain'].isna().astype(int)
feature_df['feat_amount_log1p'] = np.log1p(feature_df['TransactionAmt'])

base_feature_cols = [
    'feat_productcd_c_flag',
    'feat_high_risk_r_email_flag',
    'feat_card6_credit_flag',
    'feat_high_risk_p_email_flag',
    'feat_card4_discover_flag',
    'feat_missing_r_email_flag',
    'feat_amount_log1p',
]

train_df, valid_df = train_test_split(
    feature_df,
    test_size=0.2,
    random_state=42,
    stratify=feature_df['isFraud'],
)

train_df = train_df.copy()
valid_df = valid_df.copy()

card1_amount_stats = (
    train_df.groupby('card1')['TransactionAmt']
    .agg(card1_amt_mean='mean', card1_amt_std='std')
)

train_df = train_df.join(card1_amount_stats, on='card1').copy()
valid_df = valid_df.join(card1_amount_stats, on='card1').copy()

for current_df in (train_df, valid_df):
    current_threshold = current_df['card1_amt_mean'] + 3 * current_df['card1_amt_std'].fillna(0)
    current_df['feat_amount_gt_card1_avg_plus_3std'] = (
        current_df['TransactionAmt'] > current_threshold
    ).astype(int)

feature_cols = base_feature_cols + ['feat_amount_gt_card1_avg_plus_3std']

y_train = train_df['isFraud'].copy()
y_valid = valid_df['isFraud'].copy()

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

print('feature_cols:', feature_cols)
print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('train fraud rate:', round(100 * y_train.mean(), 3), '%')
print('valid fraud rate:', round(100 * y_valid.mean(), 3), '%')

display(X_train.head())

feature_cols: ['feat_productcd_c_flag', 'feat_high_risk_r_email_flag', 'feat_card6_credit_flag', 'feat_high_risk_p_email_flag', 'feat_card4_discover_flag', 'feat_missing_r_email_flag', 'feat_amount_log1p', 'feat_amount_gt_card1_avg_plus_3std']
X_train: (472432, 8)
X_valid: (118108, 8)
train fraud rate: 3.499 %
valid fraud rate: 3.499 %


,feat_productcd_c_flag,feat_high_risk_r_email_flag,feat_card6_credit_flag,feat_high_risk_p_email_flag,feat_card4_discover_flag,feat_missing_r_email_flag,feat_amount_log1p,feat_amount_gt_card1_avg_plus_3std
40809,0,0,1,0,0,0,4.615121,0
285886,0,0,0,0,0,1,3.433665,0
104256,0,0,1,0,0,1,4.690889,0
507860,0,0,1,0,1,1,5.492856,0
196382,0,0,0,0,0,1,4.770685,0


## First Fair Model Comparison

Теперь у нас есть один и тот же честный baseline feature set для обеих моделей.

На этом шаге сравниваем:

- `LogisticRegression`
- `RandomForestClassifier`

Что важно понять:

- обе модели обучаются на одном и том же `X_train`;
- обе модели проверяются на одном и том же `X_valid`;
- значит различия в метриках связаны именно с моделью, а не с разными данными.

Главная цель:

- понять, дает ли `RandomForest` полезный прирост по сравнению с `LogisticRegression`;
- и не забывать, что для anti-fraud важны не только общие метрики, но и бизнес-интерпретация результата.

In [10]:
logreg_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced',
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample',
)

logreg_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

logreg_pred = logreg_model.predict(X_valid)
logreg_proba = logreg_model.predict_proba(X_valid)[:, 1]

rf_pred = rf_model.predict(X_valid)
rf_proba = rf_model.predict_proba(X_valid)[:, 1]

model_metrics_df = pd.DataFrame([
    {
        'model_name': 'LogisticRegression',
        'precision': precision_score(y_valid, logreg_pred, zero_division=0),
        'recall': recall_score(y_valid, logreg_pred, zero_division=0),
        'f1': f1_score(y_valid, logreg_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, logreg_proba),
    },
    {
        'model_name': 'RandomForestClassifier',
        'precision': precision_score(y_valid, rf_pred, zero_division=0),
        'recall': recall_score(y_valid, rf_pred, zero_division=0),
        'f1': f1_score(y_valid, rf_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, rf_proba),
    },
])

display(Markdown("### model_metrics_df"))
display(model_metrics_df)

model_metrics_comparison_df = model_metrics_df.set_index('model_name').T.copy()
model_metrics_comparison_df['delta_rf_minus_logreg'] = (
    model_metrics_comparison_df['RandomForestClassifier']
    - model_metrics_comparison_df['LogisticRegression']
)

display(Markdown("### model_metrics_comparison_df"))
display(model_metrics_comparison_df)

### model_metrics_df

,model_name,precision,recall,f1,roc_auc
0,LogisticRegression,0.087011,0.609243,0.152274,0.745291
1,RandomForestClassifier,0.091049,0.614082,0.158585,0.763848


### model_metrics_comparison_df

model_name,LogisticRegression,RandomForestClassifier,delta_rf_minus_logreg
precision,0.087011,0.091049,0.004039
recall,0.609243,0.614082,0.004839
f1,0.152274,0.158585,0.006312
roc_auc,0.745291,0.763848,0.018557


## Threshold Comparison By Model

Теперь сравниваем модели не только по общим метрикам, но и по поведению на одинаковых threshold.

Это важно для anti-fraud, потому что одна модель может выглядеть лучше по `roc_auc`,
но при этом создавать слишком большой поток на `manual review`.

На этом шаге смотрим:

- как меняется `precision`
- как меняется `recall`
- как меняется `manual_review_rate_pct`

Главная цель:

- понять, какая модель удобнее для review-процесса;
- и не путать “модель с лучшим ранжированием” и “модель с лучшим операционным поведением”.

In [11]:
threshold_rows = []

model_predictions = {
    'LogisticRegression': logreg_proba,
    'RandomForestClassifier': rf_proba,
}

for model_name, current_proba in model_predictions.items():
    for threshold in [0.3, 0.5, 0.7]:
        current_pred = (current_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_valid, current_pred).ravel()

        threshold_rows.append({
            'model_name': model_name,
            'threshold': threshold,
            'precision': precision_score(y_valid, current_pred, zero_division=0),
            'recall': recall_score(y_valid, current_pred, zero_division=0),
            'f1': f1_score(y_valid, current_pred, zero_division=0),
            'predicted_fraud_count': int(current_pred.sum()),
            'manual_review_rate_pct': round(100 * current_pred.mean(), 2),
            'tp': int(tp),
            'fp': int(fp),
            'fn': int(fn),
            'tn': int(tn),
        })

threshold_df_by_model = pd.DataFrame(threshold_rows)

display(Markdown("### threshold_df_by_model"))
display(threshold_df_by_model)

manual_review_comparison_df = threshold_df_by_model.pivot(
    index='threshold',
    columns='model_name',
    values='manual_review_rate_pct',
).copy()

manual_review_comparison_df['delta_rf_minus_logreg'] = (
    manual_review_comparison_df['RandomForestClassifier']
    - manual_review_comparison_df['LogisticRegression']
)

display(Markdown("### manual_review_comparison_df"))
display(manual_review_comparison_df)

### threshold_df_by_model

,model_name,threshold,precision,recall,f1,predicted_fraud_count,manual_review_rate_pct,tp,fp,fn,tn
0,LogisticRegression,0.3,0.046464,0.880958,0.088272,78362,66.35,3641,74721,492,39254
1,LogisticRegression,0.5,0.087011,0.609243,0.152274,28939,24.50,2518,26421,1615,87554
2,LogisticRegression,0.7,0.140176,0.390273,0.206266,11507,9.74,1613,9894,2520,104081
3,RandomForestClassifier,0.3,0.050504,0.875635,0.095499,71658,60.67,3619,68039,514,45936
4,RandomForestClassifier,0.5,0.091049,0.614082,0.158585,27875,23.60,2538,25337,1595,88638
5,RandomForestClassifier,0.7,0.162319,0.418582,0.233926,10658,9.02,1730,8928,2403,105047


### manual_review_comparison_df

model_name,LogisticRegression,RandomForestClassifier,delta_rf_minus_logreg
threshold,,,
0.3,66.35,60.67,-5.68
0.5,24.50,23.60,-0.90
0.7,9.74,9.02,-0.72


## Week 3 Final Summary

На этой неделе мы закрыли baseline model этап для anti-fraud MVP.

Что сделано:
- обучен честный baseline на `LogisticRegression`;
- выполнен `threshold analysis`;
- проверены behavioral / temporal feature experiments;
- выполнено сравнение `LogisticRegression` и `RandomForestClassifier` на одном и том же feature set и одном и том же split.

Главный результат:
- `RandomForestClassifier` показал более сильный результат, чем `LogisticRegression`;
- модель дает лучшее сочетание `precision`, `recall` и `f1`;
- при этом `manual_review_rate` оказался ниже на одинаковых threshold.

Практический вывод:
- текущий лучший MVP-кандидат модели: `RandomForestClassifier`;
- для `manual review` наиболее реалистично дальше рассматривать threshold около `0.7`;
- `0.5` можно оставить как reference point для сравнения.

Что это значит для проекта:
- `Week 3` можно считать закрытой;
- дальше логично переходить к упаковке результата и подготовке MVP scoring flow.